In [1]:
import pandas as pd
import numpy as np

# 数据加载模块
try:
    # 使用完整路径加载租金训练集和测试集数据
    df_train_rent = pd.read_csv('/home/mw/input/midata1852/ruc_Class25Q2_train_rent (1).csv')
    df_test_rent = pd.read_csv('/home/mw/input/midata1852/ruc_Class25Q2_test_rent (1).csv')
    
    # 打印数据加载成功信息和数据维度
    print(f"成功加载训练集，维度: {df_train_rent.shape}")
    print(f"成功加载测试集，维度: {df_test_rent.shape}")
    
except FileNotFoundError as e:
    # 文件未找到时的错误处理
    print(f"错误：找不到数据文件: {e}")
    print("请检查文件路径是否正确")

# 数据探索分析模块
print("\n--- 训练集前5行 ---")
print(df_train_rent.head())

# 查看训练集基本信息，包括列名、数据类型、非空值数量等
print("\n--- 训练集信息 ---")
df_train_rent.info()

# 查看数值型列的基本统计信息（计数、均值、标准差、最小值、四分位数等）
print("\n--- 训练集数值统计 ---")
print(df_train_rent.describe())

# 如果有测试集，也可以查看测试集的基本信息
if 'df_test_rent' in locals():
    print("\n--- 测试集基本信息 ---")
    print(f"测试集维度: {df_test_rent.shape}")
    print("\n测试集前3行:")
    print(df_test_rent.head(3))

/tmp/ipykernel_104/3175485860.py:7: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train_rent = pd.read_csv('/home/mw/input/midata1852/ruc_Class25Q2_train_rent (1).csv')


成功加载训练集，维度: (98899, 46)
成功加载测试集，维度: (9773, 46)

--- 训练集前5行 ---
   城市      户型   装修          Price      楼层      面积 朝向        交易时间 付款方式 租赁方式  \
0   0  1室1厅1卫  精装修  654646.481811    4/6层  36.42㎡  西  2024-11-28  季付价   整租   
1   0  1室1厅1卫  精装修  665412.057415    4/6层  41.00㎡  南  2024-10-30  季付价   整租   
2   0  1室1厅1卫  精装修  778222.820548   1/18层  37.36㎡  北  2024-11-12  季付价   整租   
3   0  3室1厅2卫  精装修  612084.974699   1/10层  55.42㎡  南  2024-10-14  季付价   整租   
4   0  1室1厅1卫  精装修  994732.124864  18/18层  49.30㎡  南  2024-12-08  季付价   整租   

   ...     供水        供暖     供电            燃气费       供热费    停车位 停车费用  \
0  ...     民水      集中供暖     民电       2.61元/m³  24-30元/㎡  450.0  150   
1  ...     民水      集中供暖     民电       2.61元/m³     30元/㎡  150.0  150   
2  ...  商水/民水      集中供暖  商电/民电  2.61-2.63元/m³  30-46元/㎡  965.0  500   
3  ...     商水  集中供暖/自采暖  商电/民电       2.61元/m³  30-44元/㎡  500.0  550   
4  ...     民水      集中供暖     民电       2.61元/m³     30元/㎡  400.0  150   

      coord_x    coord_y                 

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# 数据预处理模块：特征选择和数据类型清理

# 1.1 自动计算训练集各列缺失率（与房价代码功能一致）
missing_rates = df_train_rent.isna().mean()  # 计算缺失比例（0-1）
high_missing_cols = missing_rates[missing_rates > 0.5].index.tolist()  # 筛选缺失率>50%的列
print(f"自动筛选缺失率>50%的列：{high_missing_cols}")

# 1.2 手动指定需删除的列（基于租金业务特性调整）
manual_drop_cols = [
    '客户反馈',         # 数据泄漏（含租金相关反馈）
    '物业办公电话',     # 非特征（无预测价值）
    '供水', '供电',  # 冗杂
    '租期'            # ~50% 缺失 (格式未知，暂不处理)
]

# 1.3 合并自动筛选和手动指定的删除列（与房价代码功能一致）
columns_to_drop = list(set(high_missing_cols + manual_drop_cols))
print(f"最终确定删除的列：{columns_to_drop}")
print(f"总共删除 {len(columns_to_drop)} 个特征")

# 在训练集上删除指定列，创建清理后的副本（与房价代码功能一致）
df_train_rent_cleaned = df_train_rent.drop(columns=columns_to_drop).copy()

# 在测试集上删除同样的列，确保特征一致（与房价代码功能一致）
try:
    df_test_rent_cleaned = df_test_rent.drop(columns=columns_to_drop).copy()
    print("测试集特征删除成功")
except NameError:
    print("错误：测试集数据未正确加载")

print("已完成基于缺失率分析的特征列筛选")

# 重复删除步骤确保清理彻底（与房价代码功能一致）
df_train_rent_cleaned = df_train_rent.drop(columns=columns_to_drop).copy()
try:
    df_test_rent_cleaned = df_test_rent.drop(columns=columns_to_drop).copy()
except NameError:
    print("错误：测试集数据未正确加载")

print("已完成特征列筛选，移除了数据泄漏、高缺失率和无关特征")


# 数据类型清理：处理带有单位的数值字段（根据租金数据结构调整）
# 重命名目标变量（租金场景特有步骤，房价代码无需此步）
try:
    df_train_rent_cleaned.rename(columns={'Price': 'Rent'}, inplace=True)
    print("目标变量已从'Price'重命名为'Rent'")
except Exception as e:
    print(f"重命名目标变量时出错: {e}")
# 租金数据中面积字段为"面积"，而非房价的"建筑面积""套内面积"
try:
    # 清理 '面积' - 移除"㎡"单位并转换为浮点数
    df_train_rent_cleaned['面积'] = df_train_rent_cleaned['面积'].str.replace('㎡', '').astype(float)
    df_test_rent_cleaned['面积'] = df_test_rent_cleaned['面积'].str.replace('㎡', '').astype(float)
    print("'面积' 列已清理为数值型")
except Exception as e:
    print(f"清理面积字段时出错: {e}")


# 数据集分割：分离特征和目标变量

# 分离特征(X)和目标变量(y)（与房价代码功能一致，变量名适配）
y_rent = df_train_rent_cleaned['Rent']  # 目标变量：租金
X_rent = df_train_rent_cleaned.drop(columns=['Rent'])  # 特征矩阵

# 按80/20比例分割为训练集和验证集（与房价代码功能一致）
X_train_rent, X_val_rent, y_train_rent, y_val_rent = train_test_split(
    X_rent, 
    y_rent, 
    test_size=0.2, 
    random_state=111  # 随机种子确保可复现
)

# 创建独立副本避免警告（与房价代码功能一致）
X_train_rent = X_train_rent.copy()
X_val_rent = X_val_rent.copy()

# 为测试集创建独立副本（与房价代码功能一致）
X_test_rent = df_test_rent_cleaned.copy()

print(f"数据集分割完成 - 训练集: {X_train_rent.shape}, 验证集: {X_val_rent.shape}, 测试集: {X_test_rent.shape}")


# 测试集ID列处理（与房价代码功能一致）
if 'ID' in X_test_rent.columns:
    # 存储ID用于最终提交（租金场景补充步骤，方便结果输出）
    test_ids_rent = X_test_rent['ID'].copy()
    X_test_rent = X_test_rent.drop(columns=['ID'])
    print("已删除测试集中的ID列并存储ID用于提交")


# 缺失值处理模块（与房价代码功能一致，变量名适配）

# 识别数值型和类别型特征
numerical_features = X_train_rent.select_dtypes(include=['float64', 'int64']).columns
categorical_features = X_train_rent.select_dtypes(include=['object']).columns

print(f"\n缺失值填充策略:")
print(f"将使用 '中位数' 填充 {len(numerical_features)} 个数值型特征")
print(f"将使用 'Unknown' 填充 {len(categorical_features)} 个类别型特征")

# 填充数值型特征 - 使用训练集的中位数（避免数据泄漏）
for col in numerical_features:
    median_value = X_train_rent[col].median()  # 仅用训练集计算
    X_train_rent[col] = X_train_rent[col].fillna(median_value)
    X_val_rent[col] = X_val_rent[col].fillna(median_value)
    if col in X_test_rent.columns:
        X_test_rent[col] = X_test_rent[col].fillna(median_value)

print("已完成数值型特征的缺失值填充")

# 填充类别型特征 - 使用"Unknown"标记缺失值
fill_value = "Unknown" 
for col in categorical_features:
    X_train_rent[col] = X_train_rent[col].fillna(fill_value)
    X_val_rent[col] = X_val_rent[col].fillna(fill_value)
    if col in X_test_rent.columns:
        X_test_rent[col] = X_test_rent[col].fillna(fill_value)

print("已完成类别型特征的缺失值填充")


# 类别特征分析：检查特征基数（与房价代码功能一致，变量名适配）

# 重新识别object类型列（填充后可能变化）
categorical_features = X_train_rent.select_dtypes(include=['object']).columns

print(f"\n特征分析:")
print(f"在 X_train_rent 中找到了 {len(categorical_features)} 个 object 类型的特征。")

# 计算并分析类别特征的基数（唯一值数量）
print("\n--- 类别特征的基数 (唯一值数量) 检查 ---")
cardinality = X_train_rent[categorical_features].nunique()
print(cardinality.sort_values())  # 按基数从小到大排序

# 输出最终数据维度
print(f"\n数据预处理完成 - X_train_rent 当前维度: {X_train_rent.shape}")

自动筛选缺失率>50%的列：['装修', '车位', '采暖', '环线位置', '物业办公电话', '供暖', '供热费']
最终确定删除的列：['供水', '环线位置', '装修', '租期', '采暖', '供暖', '物业办公电话', '车位', '客户反馈', '供热费', '供电']
总共删除 11 个特征
测试集特征删除成功
已完成基于缺失率分析的特征列筛选
已完成特征列筛选，移除了数据泄漏、高缺失率和无关特征
目标变量已从'Price'重命名为'Rent'
'面积' 列已清理为数值型
数据集分割完成 - 训练集: (79119, 34), 验证集: (19780, 34), 测试集: (9773, 35)
已删除测试集中的ID列并存储ID用于提交

缺失值填充策略:
将使用 '中位数' 填充 11 个数值型特征
将使用 'Unknown' 填充 23 个类别型特征
已完成数值型特征的缺失值填充
已完成类别型特征的缺失值填充

特征分析:
在 X_train_rent 中找到了 23 个 object 类型的特征。

--- 类别特征的基数 (唯一值数量) 检查 ---
租赁方式        2
电梯          3
用水          3
用电          3
燃气          3
付款方式        8
建筑结构       15
朝向         88
产权描述      164
户型        199
楼栋总数      210
物业类别      233
绿 化 率     240
燃气费       246
停车费用      257
交易时间      294
配套设施      677
建筑年代      730
楼层       1010
物 业 费    1027
房屋总数     1727
物业公司     1910
开发商      2057
dtype: int64

数据预处理完成 - X_train_rent 当前维度: (79119, 34)


In [3]:
import re
import numpy as np
import pandas as pd

# -------------------------- 修正后的通用多输出特征处理函数 --------------------------
def process_multi_feature(train_df, val_df, test_df, 
                          target_col, parse_func, new_cols, 
                          default_fill=0.0):
    """处理多输出特征（如户型、楼层），返回更新后的数据集"""
    try:
        if target_col in train_df.columns:
            # 解析并拆分多列
            train_new = train_df[target_col].apply(
                lambda x: pd.Series(parse_func(x), index=new_cols)
            )
            val_new = val_df[target_col].apply(
                lambda x: pd.Series(parse_func(x), index=new_cols)
            )
            test_new = test_df[target_col].apply(
                lambda x: pd.Series(parse_func(x), index=new_cols)
            )
            
            # 合并新列
            train_df = pd.concat([train_df, train_new], axis=1)
            val_df = pd.concat([val_df, val_new], axis=1)
            test_df = pd.concat([test_df, test_new], axis=1)
            print(f"已创建特征：{', '.join(new_cols)}")

            # 填充NaN（原函数内基础填充）
            for col in new_cols:
                median_val = train_df[col].median()
                fill_val = median_val if not np.isnan(median_val) else default_fill
                train_df[col] = train_df[col].fillna(fill_val)
                val_df[col] = val_df[col].fillna(fill_val)
                test_df[col] = test_df[col].fillna(fill_val)
            print(f"已填充 {', '.join(new_cols)} 的NaN（函数内基础填充）")
        
        else:
            print(f"跳过：'{target_col}' 列不在训练集中")

    except NameError as e:
        print(f"错误：{e}，请确保解析函数已定义")
    except KeyError as e:
        print(f"错误：列 {e} 未找到")
    except Exception as e:
        print(f"处理 {target_col} 时出错：{e}")
    return train_df, val_df, test_df


# -------------------------- 3.1 量化 '户型' --------------------------
print("\n--- 步骤 3.1 : 量化 '户型' ---")

# 户型解析函数（确保在调用前定义）
def parse_layout(text_data):
    if not isinstance(text_data, str): return (np.nan, np.nan, np.nan, np.nan)
    room_match = re.search(r'(\d+)(?:室|房间)', text_data)
    num_rooms = int(room_match.group(1)) if room_match else 0
    living_match = re.search(r'(\d+)厅', text_data)
    num_living = int(living_match.group(1)) if living_match else 0
    kitchen_match = re.search(r'(\d+)厨', text_data)
    num_kitchen = int(kitchen_match.group(1)) if kitchen_match else 0
    bath_match = re.search(r'(\d+)卫', text_data)
    num_bath = int(bath_match.group(1)) if bath_match else 0
    if num_rooms == 0 and num_living == 0 and num_kitchen == 0 and num_bath == 0:
        return (np.nan, np.nan, np.nan, np.nan)
    return (num_rooms, num_living, num_kitchen, num_bath)

# 调用多输出处理函数
X_train_rent, X_val_rent, X_test_rent = process_multi_feature(
    X_train_rent, X_val_rent, X_test_rent,
    target_col="户型",
    parse_func=parse_layout,
    new_cols=['Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths']
)


# -------------------------- 3.2 量化 '楼层' --------------------------
print("\n--- 步骤 3.2 : 量化 '楼层' ---")

# 楼层解析函数（确保在调用前定义）
def parse_rent_floor(text_data):
    if not isinstance(text_data, str) or '/' not in text_data:
        return (np.nan, 1.0)
    parts = text_data.split('/')
    if len(parts) != 2:
        return (np.nan, 1.0)
    floor_part = parts[0].strip()
    total_part = parts[1].strip()
    
    # 解析总楼层
    total_match = re.search(r'(\d+)', total_part)
    total_floors = float(total_match.group(1)) if total_match else 1.0
    total_floors = max(total_floors, 1.0)
    
    # 解析所在楼层
    floor_num = np.nan
    try:
        floor_num = float(floor_part)
    except ValueError:
        if '低' in floor_part:
            floor_num = max(1.0, round(0.25 * total_floors))
        elif '中' in floor_part:
            floor_num = max(1.0, round(0.50 * total_floors))
        elif '高' in floor_part:
            floor_num = max(1.0, round(0.75 * total_floors))
        elif '底' in floor_part:
            floor_num = 1.0
        elif '顶' in floor_part:
            floor_num = total_floors
    
    if not np.isnan(floor_num):
        floor_num = min(floor_num, total_floors)
        floor_num = max(floor_num, 1.0)
    return (floor_num, total_floors)

# 调用多输出处理函数
X_train_rent, X_val_rent, X_test_rent = process_multi_feature(
    X_train_rent, X_val_rent, X_test_rent,
    target_col="楼层",
    parse_func=parse_rent_floor,
    new_cols=['Floor_Num', 'Total_Floors_Rent']
)

# -------------------------- 新增：楼层列单独中位数填充 --------------------------
print("\n--- 步骤 3.2 补充：楼层相关列单独中位数填充 ---")
# 定义楼层相关的新列
floor_related_cols = ['Floor_Num', 'Total_Floors_Rent']

# 针对每个楼层列，用训练集中位数重新确认填充（避免遗漏）
for col in floor_related_cols:
    # 仅用训练集计算中位数（避免数据泄漏）
    train_median = X_train_rent[col].median()
    # 确保中位数有效，无效时用1.0（楼层最小为1）
    fill_val = train_median if not np.isnan(train_median) else 1.0
    
    # 重新填充所有数据集
    X_train_rent[col] = X_train_rent[col].fillna(fill_val)
    X_val_rent[col] = X_val_rent[col].fillna(fill_val)
    X_test_rent[col] = X_test_rent[col].fillna(fill_val)
    
    print(f"列 '{col}' - 训练集中位数：{fill_val:.1f}，已重新填充所有数据集")
    


--- 步骤 3.1 : 量化 '户型' ---
已创建特征：Num_Rooms, Num_LivingRooms, Num_Kitchens, Num_Baths
已填充 Num_Rooms, Num_LivingRooms, Num_Kitchens, Num_Baths 的NaN（函数内基础填充）

--- 步骤 3.2 : 量化 '楼层' ---
已创建特征：Floor_Num, Total_Floors_Rent
已填充 Floor_Num, Total_Floors_Rent 的NaN（函数内基础填充）

--- 步骤 3.2 补充：楼层相关列单独中位数填充 ---
列 'Floor_Num' - 训练集中位数：8.0，已重新填充所有数据集
列 'Total_Floors_Rent' - 训练集中位数：18.0，已重新填充所有数据集


In [4]:


# -------------------------- 修正后的通用单输出特征处理函数 --------------------------
def process_single_feature(train_df, val_df, test_df, 
                           target_col, parse_func, new_col, 
                           default_fill, extra_calc=None):
    """处理单输出特征，返回更新后的数据集（删除错误的函数检查）"""
    try:
        # 【删除错误的检查逻辑】parse_func是外部传入的全局函数，无需检查locals()
        
        if target_col in train_df.columns:
            # 应用解析函数
            train_new = train_df[target_col].apply(parse_func)
            val_new = val_df[target_col].apply(parse_func)
            
            # 处理测试集
            if target_col in test_df.columns:
                test_new = test_df[target_col].apply(parse_func)
            else:
                test_new = pd.Series(np.nan, index=test_df.index)
                print(f"警告：'{target_col}' 列不在测试集中")
            
            # 执行额外计算（如房龄=2025-年份）
            if extra_calc:
                train_new = extra_calc(train_new)
                val_new = extra_calc(val_new)
                test_new = extra_calc(test_new)
            
            # 合并新列
            train_df[new_col] = train_new
            val_df[new_col] = val_new
            test_df[new_col] = test_new
            print(f"已创建特征：{new_col}")

            # 填充NaN
            if train_df[new_col].isnull().any():
                median_val = train_df[new_col].median()
                fill_val = median_val if not np.isnan(median_val) else default_fill
                train_df[new_col] = train_df[new_col].fillna(fill_val)
                val_df[new_col] = val_df[new_col].fillna(fill_val)
                test_df[new_col] = test_df[new_col].fillna(fill_val)
                print(f"已用 {fill_val:.2f} 填充 {new_col} 的NaN")
            else:
                print(f"{new_col} 无NaN需填充")
        
        else:
            print(f"跳过：'{target_col}' 列不在训练集中")

    except NameError as e:
        print(f"错误：{e}，请确保解析函数已定义")
    except KeyError as e:
        print(f"错误：列 {e} 未找到")
    except Exception as e:
        print(f"处理 {target_col} 时出错：{e}")
    return train_df, val_df, test_df


# -------------------------- 步骤 3.3 : 量化 '建筑年代' --------------------------
print("\n--- 步骤 3.3 : 量化 '建筑年代' ---")

# 1. 先定义解析函数（必须在调用前定义）
def parse_build_year(text_data):
    if not isinstance(text_data, str): return np.nan
    numbers = re.findall(r'(\d{4})', text_data)  # 提取4位年份
    if len(numbers) == 0: return np.nan
    elif len(numbers) == 1: return float(numbers[0])
    else: return (float(numbers[0]) + float(numbers[-1])) / 2  # 多年份取平均

# 2. 调用修正后的单输出处理函数
X_train_rent, X_val_rent, X_test_rent = process_single_feature(
    X_train_rent, X_val_rent, X_test_rent,
    target_col="建筑年代",
    parse_func=parse_build_year,  # 传入已定义的解析函数
    new_col="House_Age_Rent",
    default_fill=0.0,
    extra_calc=lambda x: 2025 - x  # 计算房龄
)


--- 步骤 3.3 : 量化 '建筑年代' ---
已创建特征：House_Age_Rent
已用 17.00 填充 House_Age_Rent 的NaN


In [5]:
# -------------------------- 3.4 量化 '房屋总数' + 3.5 量化 '楼栋总数' --------------------------

# 通用数量解析函数（提取数字）
def parse_total_units(text_data):
    if not isinstance(text_data, str): return np.nan
    match = re.search(r'(\d+)', text_data)  # 提取首个数字
    if match: return float(match.group(1))
    else: return np.nan


# 3.4 量化 '房屋总数'
print("\n--- 步骤 3.4 : 量化 '房屋总数' ---")
X_train_rent, X_val_rent, X_test_rent = process_single_feature(
    X_train_rent, X_val_rent, X_test_rent,
    target_col="房屋总数",
    parse_func=parse_total_units,
    new_col="Total_Units_Rent",
    default_fill=0.0
)


# 3.5 量化 '楼栋总数'
print("\n--- 步骤 3.5: 量化 '楼栋总数' ---")
X_train_rent, X_val_rent, X_test_rent = process_single_feature(
    X_train_rent, X_val_rent, X_test_rent,
    target_col="楼栋总数",
    parse_func=parse_total_units,
    new_col="Total_Buildings_Rent",
    default_fill=0.0
)


--- 步骤 3.4 : 量化 '房屋总数' ---
已创建特征：Total_Units_Rent
已用 1445.00 填充 Total_Units_Rent 的NaN

--- 步骤 3.5: 量化 '楼栋总数' ---
已创建特征：Total_Buildings_Rent
已用 13.00 填充 Total_Buildings_Rent 的NaN


In [6]:

# -------------------------- 3.6 量化 '物 业 费' + 3.7 量化 '绿 化 率' + 3.8 量化 '燃气费' --------------------------

# 通用费用/比例解析函数（提取数字并平均）
def parse_property_fee(text_data):
    if not isinstance(text_data, str): return np.nan
    numbers = re.findall(r'(\d+\.?\d*)', text_data)  # 提取数字（支持小数）
    numbers = [n for n in numbers if n]  # 过滤空值
    if len(numbers) == 0: return np.nan
    elif len(numbers) == 1: return float(numbers[0])
    else: return (float(numbers[0]) + float(numbers[-1])) / 2  # 多数字取平均


# 3.6 量化 '物 业 費'
print("\n--- 步骤 3.6 : 量化 '物 业 费' ---")
X_train_rent, X_val_rent, X_test_rent = process_single_feature(
    X_train_rent, X_val_rent, X_test_rent,
    target_col="物 业 费",
    parse_func=parse_property_fee,
    new_col="Property_Fee_Rent",
    default_fill=0.0
)


# 3.7 量化 '绿 化 率'（额外转换为比例）
print("\n--- 步骤 3.7 : 量化 '绿 化 率' ---")
X_train_rent, X_val_rent, X_test_rent = process_single_feature(
    X_train_rent, X_val_rent, X_test_rent,
    target_col="绿 化 率",
    parse_func=parse_property_fee,
    new_col="Greening_Rate_Rent",
    default_fill=0.3,  # 绿化率默认值
    extra_calc=lambda x: x / 100  # 转换为比例（如30%→0.3）
)


# 3.8 量化 '燃气费'
print("\n--- 步骤 3.8 : 量化 '燃气费' ---")
X_train_rent, X_val_rent, X_test_rent = process_single_feature(
    X_train_rent, X_val_rent, X_test_rent,
    target_col="燃气费",
    parse_func=parse_property_fee,
    new_col="Gas_Fee_Rent",
    default_fill=2.5  # 燃气费默认值
)


--- 步骤 3.6 : 量化 '物 业 费' ---
已创建特征：Property_Fee_Rent
已用 2.20 填充 Property_Fee_Rent 的NaN

--- 步骤 3.7 : 量化 '绿 化 率' ---
已创建特征：Greening_Rate_Rent
已用 0.35 填充 Greening_Rate_Rent 的NaN

--- 步骤 3.8 : 量化 '燃气费' ---
已创建特征：Gas_Fee_Rent
已用 2.95 填充 Gas_Fee_Rent 的NaN


In [7]:
# -------------------------- 3.9 量化 '停车费用' --------------------------

# 停车费硬编码表（特殊规则）
hardcoded_parking_fees = {
    "一元钱一小时，单次24小时内最高12元一次": 12.0 * 30,
    "小区没有停车费": 0.0, "无固定车位不收费": 0.0,
    "售价:30万/位；租价450元/位/月": 450.0,
    "每小时2元每个月70元": 70.0, "每小时2元/位 , 每个月50元/位": 50.0,
    "露天250元/月/位，3元/时/位；室内500元/月/位，3元/时/位": np.mean([250.0, 500.0]),
    "露天16元/小时，室内700/月": 700.0,
    "临保2.5元/小时月保400元/月": 400.0, "临保2.5元/时/位，月保400元/月/位": 400.0,
    "临保:4元/小时/位,月保:550元/位/月": 550.0,
    "固定车位1400元/年，非固定车位100元/月": np.mean([1400.0 / 12.0, 100.0]),
    "第一小时5块，后面1小时1块，一天15封顶": 15.0 * 30,
    "地下400，地上免费": np.mean([400.0, 0.0]),
    "地下350元/月/位 加60管理费/月": 350.0 + 60.0,
    "地上免费  地下400元/月": np.mean([0.0, 400.0]),
    "地上4元/小时/位": 4.0 * 8.0 * 30.0,
    "地上150元/月/位，地下2元/时/位，地下固定车位450元/月/位": np.mean([150.0, 2.0 * 8.0 * 30.0, 450.0])
}

# 停车费解析函数（含硬编码逻辑）
def parse_parking_fee_hardcoded(text_data):
    if not isinstance(text_data, str): return np.nan
    text_original = text_data.strip()
    text_lower = text_original.lower()
    
    # 优先匹配硬编码表
    if text_original in hardcoded_parking_fees:
        return hardcoded_parking_fees[text_original]
    
    # 明确免费场景
    zero_keywords = ['免费', '没有停车费', '不收费']
    if any(keyword in text_lower for keyword in zero_keywords) or text_lower in ['无', '暂无', '0']:
        return 0.0
    
    # 明确未知场景
    unknown_keywords = ['unknown', '未知', '无法核实', '无法获知']
    if any(keyword in text_lower for keyword in unknown_keywords):
        return np.nan
    
    # 最后尝试提取数字平均
    numbers = re.findall(r'(\d+\.?\d*)', text_original)
    numbers = [float(n) for n in numbers if n]
    if len(numbers) > 0:
        return np.mean(numbers)
    return np.nan

# 调用单输出处理函数
X_train_rent, X_val_rent, X_test_rent = process_single_feature(
    X_train_rent, X_val_rent, X_test_rent,
    target_col="停车费用",
    parse_func=parse_parking_fee_hardcoded,
    new_col="Parking_Fee_Rent",
    default_fill=300.0  # 停车费默认值
)

已创建特征：Parking_Fee_Rent
已用 300.00 填充 Parking_Fee_Rent 的NaN


In [8]:
print("--- 步骤 3.10 ：创建 Polynomial Features ---")

# 创建 '面积' 的平方项（使用租金数据集变量名）
X_train_rent['Area_sq'] = X_train_rent['面积'] ** 2
X_val_rent['Area_sq'] = X_val_rent['面积'] ** 2
X_test_rent['Area_sq'] = X_test_rent['面积'] ** 2

print("已成功创建 'Area_sq' (面积的平方) 特征。")

# 检查结果
check_cols = ['面积', 'Area_sq']
print("\n--- 新的 Area_sq 特征 (前5行) ---")
print(X_train_rent[check_cols].head())

# 更新维度
print(f"\nX_train_rent 新维度: {X_train_rent.shape}")

--- 步骤 3.10 ：创建 Polynomial Features ---
已成功创建 'Area_sq' (面积的平方) 特征。

--- 新的 Area_sq 特征 (前5行) ---
           面积     Area_sq
66850   40.16   1612.8256
71707   88.00   7744.0000
61913  132.00  17424.0000
35080   92.00   8464.0000
3691    42.12   1774.0944

X_train_rent 新维度: (79119, 48)


In [9]:
import matplotlib.pyplot as plt
import warnings
import numpy as np
import seaborn as sns
import pandas as pd

# 补充：设置避免警告和无限值处理（与房价版保持一致）
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option('mode.use_inf_as_na', True)

print("--- 步骤 3.11 (租金版): 对目标变量 (Rent) 进行 Log Transformation ---")

try:
    # --- 1. 核心：log1p 对数转换 ---
    y_train_rent_log = np.log1p(y_train_rent)  # 训练集租金转换
    y_val_rent_log = np.log1p(y_val_rent)      # 验证集租金转换
    print("Rent 目标变量已成功进行 log1p 转换。")

    # --- 2. 可视化对比：参照房价版优化（加数据点标注+调整颜色样式） ---
    plt.figure(figsize=(12, 5))

    # 子图1：原始租金分布（优化：清理数据+颜色+边框+数据点标注）
    plt.subplot(1, 2, 1)
    # 清理异常值/空值，避免绘图报错
    train_rent_clean = y_train_rent.replace([np.inf, -np.inf], np.nan).dropna()
    sns.histplot(
        train_rent_clean,
        kde=True,
        color="#f39c12",  # 原始租金用橙色，视觉醒目且区分房价版
        edgecolor="white",  # 白色边框避免柱子粘连
        line_kws={"color": "#e67e22", "linewidth": 2}  # 密度曲线用深橙色，更清晰
    )
    # 叠加数据点标注，显示原始租金的密集位置
    sns.rugplot(train_rent_clean, color="#f39c12", alpha=0.5)
    plt.title('Original Rent Distribution (y_train_rent)')
    plt.xlabel('Rent (Original, e.g., 3000 yuan/month)')  # 保留原租金单位标注

    # 子图2：转换后对数租金分布（优化：清理数据+对比色+边框+数据点标注）
    plt.subplot(1, 2, 2)
    # 清理异常值/空值
    train_rent_log_clean = y_train_rent_log.replace([np.inf, -np.inf], np.nan).dropna()
    sns.histplot(
        train_rent_log_clean,
        kde=True,
        color="#1abc9c",  # 转换后用青色，与原始租金的橙色形成对比
        edgecolor="white",  # 白色边框避免柱子粘连
        line_kws={"color": "#16a085", "linewidth": 2}  # 密度曲线用深青色，更清晰
    )
    # 叠加数据点标注，显示对数租金的密集位置
    sns.rugplot(train_rent_log_clean, color="#1abc9c", alpha=0.5)
    plt.title('Log(1+Rent) Distribution (y_train_rent_log)')
    plt.xlabel('Log(1+Rent)')  # 保留原x轴标签

    plt.tight_layout()  # 调整布局防止重叠
    plt.show()

    # --- 3. 保留：计算全局均值（用于基准模型/平滑） ---
    global_mean_rent_log = y_train_rent_log.mean()
    print(f"\n全局平均对数租金 (Global Mean for Smoothing): {global_mean_rent_log:.4f}")
    print("→ 用途：可作为“基准模型”（预测所有样本为均值），评估后续模型是否更优。")

    # --- 4. 保留：逆转换提醒（与房价版一致） ---
    print("\n⚠️  重要：模型预测后需用 np.expm1() 逆转换（e^x - 1），还原真实租金！")

# 异常1：变量未定义（如分割步骤没跑，y_train_rent不存在）
except NameError as e:
    print(f"🚨 错误：变量未找到 ({e})。请确保分割步骤已成功运行！")

# 异常2：其他未知错误（如租金数据异常）
except Exception as e:
    print(f"🚨 对数转换时发生意外错误: {e}")
    print("请检查 y_train_rent/y_val_rent 数据：是否有非数值、负数或极端值！")

--- 步骤 3.11 (租金版): 对目标变量 (Rent) 进行 Log Transformation ---
Rent 目标变量已成功进行 log1p 转换。


<Figure size 864x360 with 2 Axes>


全局平均对数租金 (Global Mean for Smoothing): 12.9589
→ 用途：可作为“基准模型”（预测所有样本为均值），评估后续模型是否更优。

⚠️  重要：模型预测后需用 np.expm1() 逆转换（e^x - 1），还原真实租金！


In [10]:
import pandas as pd
import numpy as np
import sys  # 用于异常处理


# ---------------------- 配置化参数（集中管理，便于调参） ----------------------
RENT_CONFIG = {
    "smoothing_factor": 20,  # 全局平滑因子（替代原m_city/m_district/m_block硬编码）
    "target_col_default": "Rent",  # 目标列默认名
    "original_id_cols": ["城市", "区县", "板块"],  # 原始类别列（后续需删除）
    "encoded_cols": {  # 编码后新列名映射
        "城市": "TE_City_Rent",
        "区县": "TE_District_Rent",
        "板块": "TE_Block_Stratified_Rent"
    }
}


# ---------------------- 工具函数封装（减少重复代码） ----------------------
def get_train_data_for_encoding(X_train: pd.DataFrame, y_train_log: pd.Series) -> pd.DataFrame:
    """合并训练集特征与对数目标变量，校验索引一致性"""
    # 关键校验：避免因索引不一致导致的统计偏差
    if not X_train.index.equals(y_train_log.index):
        raise ValueError("❌ 训练集特征与目标变量索引不一致，会导致编码统计错误！")
    
    train_data = pd.concat([X_train, y_train_log], axis=1)
    print(f"✅ 合并后训练集形状: {train_data.shape}（特征+目标变量）")
    return train_data


def apply_encoding_to_datasets(
    X_train: pd.DataFrame,
    X_val: pd.DataFrame,
    X_test: pd.DataFrame,
    encode_col: str,  # 待编码的原始列（如"城市"）
    encode_map: dict,  # 编码映射字典
    fill_value: float,  # 缺失值填充值（全局均值）
    new_col_name: str  # 编码后新列名
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """统一将编码映射应用到训练/验证/测试集，返回处理后的数据"""
    # 训练集：直接映射（理论无缺失）
    X_train[new_col_name] = X_train[encode_col].map(encode_map)
    # 验证集/测试集：映射后用全局均值填充新类别
    X_val[new_col_name] = X_val[encode_col].map(encode_map).fillna(fill_value)
    X_test[new_col_name] = X_test[encode_col].map(encode_map).fillna(fill_value)
    
    # 打印缺失值统计（便于调试）
    print(f"  - 训练集{new_col_name}缺失值: {X_train[new_col_name].isnull().sum()}")
    print(f"  - 验证集{new_col_name}缺失值: {X_val[new_col_name].isnull().sum()}")
    print(f"  - 测试集{new_col_name}缺失值: {X_test[new_col_name].isnull().sum()}")
    
    return X_train, X_val, X_test


# ---------------------- 主流程：租金目标编码 ----------------------
print("\n" + "="*50)
print("--- 步骤 3.12 : 执行目标编码 (城市/区县/板块) ---")
print("="*50)

# ---------------------- 1. 数据准备与校验 ----------------------
print("\n" + "-"*40)
print("--- 阶段1：数据准备与目标变量校验 ---")
print("-"*40)

# 确定目标列名
target_col_rent_name = y_train_rent_log.name if y_train_rent_log.name is not None else RENT_CONFIG["target_col_default"]
print(f"✅ 目标变量列名: {target_col_rent_name}")

# 合并训练集特征与目标变量（带校验）
try:
    train_data_for_encoding_rent = get_train_data_for_encoding(X_train_rent, y_train_rent_log)
except NameError as e:
    print(f"❌ 错误：未找到变量{e}，请确保X_train_rent和y_train_rent_log已定义")
    sys.exit(1)

# 校验目标变量无负值（避免log转换异常）
if (np.expm1(y_train_rent_log) < 0).any():  # 还原为原始租金值检查
    raise ValueError("❌ 租金目标变量存在负值，不符合业务逻辑！请检查原始数据")

# 计算全局均值（平滑基准）
global_mean_rent_log = y_train_rent_log.mean()
print(f"✅ 全局平均对数租金（平滑基准）: {global_mean_rent_log:.6f}")


# ---------------------- 2. 城市目标编码 ----------------------
print("\n" + "-"*40)
print(f"--- 阶段2：{RENT_CONFIG['encoded_cols']['城市']} 编码 ---")
print("-"*40)

# 计算城市统计信息（均值+样本量）
city_stats_rent = train_data_for_encoding_rent.groupby("城市")[target_col_rent_name].agg(["mean", "count"]).reset_index()
print(f"✅ 训练集中包含{len(city_stats_rent)}个城市，示例前3个统计:")
print(city_stats_rent.head(3).to_string(index=False))  # 打印示例便于校验

# 计算平滑编码值（使用配置的平滑因子）
city_stats_rent[RENT_CONFIG["encoded_cols"]["城市"]] = (
    city_stats_rent["count"] * city_stats_rent["mean"] 
    + RENT_CONFIG["smoothing_factor"] * global_mean_rent_log
) / (city_stats_rent["count"] + RENT_CONFIG["smoothing_factor"])

# 生成编码映射
city_encoding_map_rent = dict(zip(city_stats_rent["城市"], city_stats_rent[RENT_CONFIG["encoded_cols"]["城市"]]))
print(f"✅ 生成{len(city_encoding_map_rent)}个城市的编码映射")

# 应用编码到三个数据集
X_train_rent, X_val_rent, X_test_rent = apply_encoding_to_datasets(
    X_train=X_train_rent,
    X_val=X_val_rent,
    X_test=X_test_rent,
    encode_col="城市",
    encode_map=city_encoding_map_rent,
    fill_value=global_mean_rent_log,
    new_col_name=RENT_CONFIG["encoded_cols"]["城市"]
)


# ---------------------- 3. 区县目标编码 ----------------------
print("\n" + "-"*40)
print(f"--- 阶段3：{RENT_CONFIG['encoded_cols']['区县']} 编码 ---")
print("-"*40)

# 计算区县统计信息
district_stats_rent = train_data_for_encoding_rent.groupby("区县")[target_col_rent_name].agg(["mean", "count"]).reset_index()
print(f"✅ 训练集中包含{len(district_stats_rent)}个区县")

# 计算平滑编码值
district_stats_rent[RENT_CONFIG["encoded_cols"]["区县"]] = (
    district_stats_rent["count"] * district_stats_rent["mean"] 
    + RENT_CONFIG["smoothing_factor"] * global_mean_rent_log
) / (district_stats_rent["count"] + RENT_CONFIG["smoothing_factor"])

# 生成编码映射
district_encoding_map_rent = dict(zip(district_stats_rent["区县"], district_stats_rent[RENT_CONFIG["encoded_cols"]["区县"]]))
print(f"✅ 生成{len(district_encoding_map_rent)}个区县的编码映射")

# 应用编码到三个数据集
X_train_rent, X_val_rent, X_test_rent = apply_encoding_to_datasets(
    X_train=X_train_rent,
    X_val=X_val_rent,
    X_test=X_test_rent,
    encode_col="区县",
    encode_map=district_encoding_map_rent,
    fill_value=global_mean_rent_log,
    new_col_name=RENT_CONFIG["encoded_cols"]["区县"]
)


# ---------------------- 4. 板块目标编码（分层平滑） ----------------------
print("\n" + "-"*40)
print(f"--- 阶段4：{RENT_CONFIG['encoded_cols']['板块']} 编码（分层平滑） ---")
print("-"*40)

# 计算板块统计信息（按"区县+板块"分组）
block_stats_rent = train_data_for_encoding_rent.groupby(["区县", "板块"])[target_col_rent_name].agg(["mean", "count"])
print(f"✅ 训练集中包含{len(block_stats_rent)}个板块（区县+板块组合）")

# 获取区县原始均值（用于分层平滑的父级基准）
try:
    district_original_mean_map_rent = dict(zip(district_stats_rent["区县"], district_stats_rent["mean"]))
except NameError:
    print("❌ 错误：未找到区县统计数据，请先执行区县编码步骤")
    sys.exit(1)


# 分层平滑函数（复用逻辑，参数通过配置传递）
def stratified_smooth_rent(row: pd.Series) -> float:
    """板块分层平滑：用父级区县均值作为基准，提升地理关联性"""
    block_mean = row["mean"]  # 板块自身均值
    block_count = row["count"]  # 板块样本量
    district_id = row.name[0]  # 从MultiIndex提取父级区县
    # 父级均值：优先用区县原始均值，无则回退到全局均值
    parent_mean = district_original_mean_map_rent.get(district_id, global_mean_rent_log)
    
    # 平滑公式（使用全局配置的平滑因子）
    return (block_count * block_mean + RENT_CONFIG["smoothing_factor"] * parent_mean) / (block_count + RENT_CONFIG["smoothing_factor"])


# 计算板块平滑编码值
block_stats_rent[RENT_CONFIG["encoded_cols"]["板块"]] = block_stats_rent.apply(stratified_smooth_rent, axis=1)
block_encoding_map_rent = block_stats_rent[RENT_CONFIG["encoded_cols"]["板块"]].to_dict()
print(f"✅ 生成{len(block_encoding_map_rent)}个板块的分层平滑编码映射")


# 板块编码应用（三级回退逻辑）
def apply_block_encoding_rent(row: pd.Series) -> float:
    """三级回退：板块→区县→全局均值，确保无缺失"""
    key = (row["区县"], row["板块"])
    # 1. 优先匹配板块
    if key in block_encoding_map_rent:
        return block_encoding_map_rent[key]
    # 2. 回退到区县编码
    elif row["区县"] in district_encoding_map_rent:
        return district_encoding_map_rent[row["区县"]]
    # 3. 最终回退到全局均值
    else:
        return global_mean_rent_log


# 应用板块编码到三个数据集
X_train_rent[RENT_CONFIG["encoded_cols"]["板块"]] = X_train_rent.apply(apply_block_encoding_rent, axis=1)
X_val_rent[RENT_CONFIG["encoded_cols"]["板块"]] = X_val_rent.apply(apply_block_encoding_rent, axis=1)
X_test_rent[RENT_CONFIG["encoded_cols"]["板块"]] = X_test_rent.apply(apply_block_encoding_rent, axis=1)

# 校验缺失值
print(f"✅ 板块编码应用完成:")
print(f"  - 训练集{RENT_CONFIG['encoded_cols']['板块']}缺失值: {X_train_rent[RENT_CONFIG['encoded_cols']['板块']].isnull().sum()}")
print(f"  - 验证集{RENT_CONFIG['encoded_cols']['板块']}缺失值: {X_val_rent[RENT_CONFIG['encoded_cols']['板块']].isnull().sum()}")
print(f"  - 测试集{RENT_CONFIG['encoded_cols']['板块']}缺失值: {X_test_rent[RENT_CONFIG['encoded_cols']['板块']].isnull().sum()}")


# ---------------------- 5. 最终清理与校验 ----------------------
print("\n" + "-"*40)
print("--- 阶段5：清理原始类别列与最终校验 ---")
print("-"*40)

# 删除原始类别列（避免模型使用原始ID）
cols_to_drop = [col for col in RENT_CONFIG["original_id_cols"] if col in X_train_rent.columns]
if cols_to_drop:
    X_train_rent = X_train_rent.drop(columns=cols_to_drop).copy()
    X_val_rent = X_val_rent.drop(columns=cols_to_drop).copy()
    X_test_rent = X_test_rent.drop(columns=cols_to_drop).copy()
    print(f"✅ 已删除原始类别列: {cols_to_drop}")
else:
    print("ℹ️  未找到需删除的原始类别列（可能已提前清理）")

# 最终维度校验
print(f"\n编码后数据集维度:")
print(f"  - 训练集: {X_train_rent.shape}")
print(f"  - 验证集: {X_val_rent.shape}")
print(f"  - 测试集: {X_test_rent.shape}")

print("\n" + "="*50)
print("--- 租金版目标编码全流程完成 ---")
print("="*50)


--- 步骤 3.12 : 执行目标编码 (城市/区县/板块) ---

----------------------------------------
--- 阶段1：数据准备与目标变量校验 ---
----------------------------------------
✅ 目标变量列名: Rent
✅ 合并后训练集形状: (79119, 49)（特征+目标变量）
✅ 全局平均对数租金（平滑基准）: 12.958874

----------------------------------------
--- 阶段2：TE_City_Rent 编码 ---
----------------------------------------
✅ 训练集中包含12个城市，示例前3个统计:
 城市      mean  count
  0 13.715371  11695
  1 12.237329   6219
  2 12.364316   9636
✅ 生成12个城市的编码映射
  - 训练集TE_City_Rent缺失值: 0
  - 验证集TE_City_Rent缺失值: 0
  - 测试集TE_City_Rent缺失值: 0

----------------------------------------
--- 阶段3：TE_District_Rent 编码 ---
----------------------------------------
✅ 训练集中包含103个区县
✅ 生成103个区县的编码映射
  - 训练集TE_District_Rent缺失值: 0
  - 验证集TE_District_Rent缺失值: 0
  - 测试集TE_District_Rent缺失值: 0

----------------------------------------
--- 阶段4：TE_Block_Stratified_Rent 编码（分层平滑） ---
----------------------------------------
✅ 训练集中包含955个板块（区县+板块组合）
✅ 生成955个板块的分层平滑编码映射
✅ 板块编码应用完成:
  - 训练集TE_Block_Stratified_Rent缺失值: 0
  - 验证集TE_

In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# ---------------------- 配置化参数（集中管理关键参数） ----------------------
RENT_CONFIG = {
    "winsorize_quantile": (0.01, 0.99),  # 缩尾分位数（替代硬编码）
    "expected_initial_cols": 33,  # 清理后预期列数（便于校验）
    "expected_final_cols": "几百列"  # 最终建模维度预期
}


# ---------------------- 工具函数封装（减少重复代码） ----------------------
def remove_duplicate_cols(df: pd.DataFrame, df_name: str) -> pd.DataFrame:
    """删除数据集中的重复列，返回处理后的数据"""
    duplicate_cols = df.columns[df.columns.duplicated()].tolist()
    if duplicate_cols:
        print(f"⚠️  {df_name}存在{len(duplicate_cols)}个重复列，已删除重复项：{duplicate_cols}")
        return df.loc[:, ~df.columns.duplicated(keep='first')]
    else:
        print(f"✅  {df_name}无重复列，无需处理")
        return df


print("\n--- 步骤 3.13 : 严格清理 + OHE + Winsorize + Scale ---")

# --- 1. 定义最终要保留的核心数值/量化/TE特征 (共 25 个) ---
final_numeric_cols_to_keep_rent = [
    '面积', 'lon', 'lat', '容 积 率', '停车位', 
    'Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths',
    'Floor_Num', 'Total_Floors_Rent',  # 保留的楼层特征
    'House_Age_Rent', 'Total_Units_Rent', 'Total_Buildings_Rent', 
    'Property_Fee_Rent', 'Greening_Rate_Rent', 'Gas_Fee_Rent', 'Parking_Fee_Rent',
    'TE_City_Rent', 'TE_District_Rent', 'TE_Block_Stratified_Rent', 
]
# 条件添加Area_sq_Rent（保留原有检查逻辑）
if 'Area_sq' in X_train_rent.columns:
    final_numeric_cols_to_keep_rent.append('Area_sq')
else:
    print("警告：'Area_sq' 未创建，未加入数值列白名单。")


# --- 2. 定义必须删除的object列 ---
object_cols_to_drop_rent = [
    # 已量化列（必须删除）
    '户型', '楼层', '建筑年代', '房屋总数', '楼栋总数', 
    '物 业 费', '绿 化 率', '燃气费', '停车费用', '交易时间',
    # 高基数列（主动删除）
    '开发商', '物业公司', '配套设施', '物业类别', '产权描述' 
]


# --- 3. 识别待OHE的低基数类别列 ---
try:
    if 'X_train_rent' not in locals():
        raise NameError("X_train_rent (包含所有object列) 未定义")
    
    # 提取当前所有object列并筛选
    current_object_cols = X_train_rent.select_dtypes(include=['object']).columns.tolist()
    categorical_cols_for_ohe_rent = [
        col for col in current_object_cols if col not in object_cols_to_drop_rent
    ]
    print(f"最终将对 {len(categorical_cols_for_ohe_rent)} 个低基数类别列进行OHE: {categorical_cols_for_ohe_rent}")
    print(f"（预期列：['朝向', '付款方式', '租赁方式', '电梯', '用水', '用电', '燃气', '建筑结构']）")


    # --- 4. 执行最终清理（白名单筛选） ---
    all_final_cols_rent = final_numeric_cols_to_keep_rent + categorical_cols_for_ohe_rent
    
    # 检查列存在性（保留原有校验）
    missing_in_train = [col for col in all_final_cols_rent if col not in X_train_rent.columns]
    if missing_in_train:
        raise KeyError(f"以下列在X_train_rent中缺失: {missing_in_train}")

    # 筛选列并创建清理后数据集
    X_train_cleaned_rent = X_train_rent[all_final_cols_rent].copy()
    X_val_cleaned_rent = X_val_rent[all_final_cols_rent].copy()
    X_test_rent_cleaned = X_test_rent[all_final_cols_rent].copy()
    
    print("最终清理完成。")
    print(f"清理后维度 (编码前): {X_train_cleaned_rent.shape}")
   

    # --- 5. 最终One-Hot编码 + 重复列处理 + 对齐 ---
    print(f"正在执行最终OHE (drop_first=True)...")
    # 执行OHE（保留原有参数）
    X_train_final_rent = pd.get_dummies(
        X_train_cleaned_rent, 
        columns=categorical_cols_for_ohe_rent, 
        dummy_na=False, 
        dtype=int, 
        drop_first=True
    )
    X_val_final_rent = pd.get_dummies(
        X_val_cleaned_rent, 
        columns=categorical_cols_for_ohe_rent, 
        dummy_na=False, 
        dtype=int, 
        drop_first=True
    )
    X_test_rent_final = pd.get_dummies(
        X_test_rent_cleaned, 
        columns=categorical_cols_for_ohe_rent, 
        dummy_na=False, 
        dtype=int, 
        drop_first=True
    )

    # 新增：删除OHE可能产生的重复列（复用工具函数）
    X_train_final_rent = remove_duplicate_cols(X_train_final_rent, "X_train_final_rent")
    X_val_final_rent = remove_duplicate_cols(X_val_final_rent, "X_val_final_rent")
    X_test_rent_final = remove_duplicate_cols(X_test_rent_final, "X_test_rent_final")

    # 列对齐（保留原有逻辑）
    final_train_columns_rent = X_train_final_rent.columns
    X_val_final_rent = X_val_final_rent.reindex(columns=final_train_columns_rent, fill_value=0)
    X_test_rent_final = X_test_rent_final.reindex(columns=final_train_columns_rent, fill_value=0)
    print("OHE、重复列处理、对齐完成。")
    print(f"最终OHE后维度: {X_train_final_rent.shape}")


    # --- 6. Winsorization（缩尾） ---
    print("\n正在执行Winsorization...")
    # 从配置获取分位数（替代硬编码）
    lower_quantile, upper_quantile = RENT_CONFIG["winsorize_quantile"]
    # 筛选需缩尾的列（保留原有逻辑）
    cols_to_winsorize_present_rent = [
        col for col in final_numeric_cols_to_keep_rent 
        if col in X_train_final_rent.columns
    ]
    print(f"将对{len(cols_to_winsorize_present_rent)}个数值列进行缩尾（分位数：{lower_quantile}-{upper_quantile}）")

    # 计算缩尾界限
    bounds = {}
    for col in cols_to_winsorize_present_rent:
        q1 = X_train_final_rent[col].quantile(lower_quantile)
        q99 = X_train_final_rent[col].quantile(upper_quantile)
        if q1 < q99:
            bounds[col] = (q1, q99)

    # 应用缩尾（保留原有逻辑）
    X_train_winsorized_rent = X_train_final_rent.copy()
    X_val_winsorized_rent = X_val_final_rent.copy()
    X_test_rent_winsorized = X_test_rent_final.copy()
    for col in bounds: 
        q1, q99 = bounds[col]
        X_train_winsorized_rent[col] = X_train_winsorized_rent[col].clip(lower=q1, upper=q99)
        X_val_winsorized_rent[col] = X_val_winsorized_rent[col].clip(lower=q1, upper=q99)
        X_test_rent_winsorized[col] = X_test_rent_winsorized[col].clip(lower=q1, upper=q99)
    print("Winsorization完成。")


    # --- 7. 特征缩放（Scale） ---
    print("\n正在执行特征缩放...")
    scaler_rent = StandardScaler()
    X_train_scaled_rent = scaler_rent.fit_transform(X_train_winsorized_rent) 
    X_val_scaled_rent = scaler_rent.transform(X_val_winsorized_rent)     
    X_test_scaled_rent = scaler_rent.transform(X_test_rent_winsorized) 
    
    # 转换为DataFrame（保留原有逻辑）
    final_columns_rent = X_train_winsorized_rent.columns 
    X_train_scaled_df_rent = pd.DataFrame(
        X_train_scaled_rent, 
        index=X_train_winsorized_rent.index, 
        columns=final_columns_rent
    )
    X_val_scaled_df_rent = pd.DataFrame(
        X_val_scaled_rent, 
        index=X_val_winsorized_rent.index, 
        columns=final_columns_rent
    )
    X_test_scaled_df_rent = pd.DataFrame(
        X_test_scaled_rent, 
        index=X_test_rent_winsorized.index, 
        columns=final_columns_rent
    )
    print("Scaling完成。")


    # 最终结果提示（优化日志）
    print("\n--- 租金数据最终准备完成 (已降维)！---")
    print(f"最终用于建模的特征维度: {X_train_scaled_df_rent.shape}")

except NameError as e:
    print(f"🚨 错误：变量未找到 ({e})。请确保TE和量化步骤已成功运行。")
except KeyError as e:
    print(f"🚨 错误：列 '{e}' 缺失。请检查白名单或之前的步骤。")
except Exception as e:
    print(f"处理过程中发生意外错误: {e}")


--- 步骤 3.13 : 严格清理 + OHE + Winsorize + Scale ---
最终将对 8 个低基数类别列进行OHE: ['朝向', '付款方式', '租赁方式', '电梯', '用水', '用电', '燃气', '建筑结构']
（预期列：['朝向', '付款方式', '租赁方式', '电梯', '用水', '用电', '燃气', '建筑结构']）
最终清理完成。
清理后维度 (编码前): (79119, 30)
正在执行最终OHE (drop_first=True)...
✅  X_train_final_rent无重复列，无需处理
✅  X_val_final_rent无重复列，无需处理
✅  X_test_rent_final无重复列，无需处理
OHE、重复列处理、对齐完成。
最终OHE后维度: (79119, 139)

正在执行Winsorization...
将对22个数值列进行缩尾（分位数：0.01-0.99）
Winsorization完成。

正在执行特征缩放...
Scaling完成。

--- 租金数据最终准备完成 (已降维)！---
最终用于建模的特征维度: (79119, 139)


In [26]:
# --- 租金步骤 4.1: OLS 建模 ---
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # 新增r2_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error # <-- Added this import
import numpy as np
import pandas as pd
import warnings

# --- 1. 评估函数 (如果需要重新定义) ---
# (确保 evaluate_model_safe 已定义，它处理 log->orig 转换和溢出)
try:
    evaluate_model_safe
except NameError:
    print("重新定义 evaluate_model_safe 函数...")
    def evaluate_model_safe(y_true_log, y_pred_log, model_name="Model"):
        mae, rmse = np.nan, np.nan
        y_true_orig = np.expm1(y_true_log)
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            try:
                y_pred_orig = np.expm1(y_pred_log)
                if not np.all(np.isfinite(y_pred_orig)): raise ValueError("Overflow/NaN after expm1")
                # Ensure metrics functions are available
                mae = mean_absolute_error(y_true_orig, y_pred_orig)
                rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
            except (OverflowError, ValueError) as e: print(f"🚨 错误：计算 {model_name} 原始尺度指标时发生溢出或无效值！将返回 NaN。")
            except NameError: print("🚨 错误: 'mean_absolute_error' or 'mean_squared_error' not imported!")
        print(f"--- {model_name} Performance ---")
        if not np.isnan(mae): print(f"MAE (原始租金): {mae:,.2f}"); print(f"RMSE (原始租金): {rmse:,.2f}")
        else: print(f"MAE (原始租金): N/A"); print(f"RMSE (原始租金): N/A")
        mae_log = mean_absolute_error(y_true_log, y_pred_log); rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
        print(f"MAE (对数尺度): {mae_log:.4f}"); print(f"RMSE (对数尺度): {rmse_log:.4f}")
        return mae, rmse


# --- 2. 初始化并训练 OLS 模型 ---
ols_model_rent = LinearRegression()
print("正在训练 OLS 模型 (使用 139 个特征)...")
ols_model_rent.fit(X_train_scaled_df_rent, y_train_rent_log)
print("OLS 模型训练完毕。")

# --- 3. 样本内评估 ---
print("\n--- (A) 样本内 (In-sample) 评估 (Rent OLS) ---")
y_train_pred_log_ols_rent = ols_model_rent.predict(X_train_scaled_df_rent)
# 新增：计算样本内R²（基于对数尺度，与模型训练目标一致）
r2_in_ols_rent = r2_score(y_train_rent_log, y_train_pred_log_ols_rent)
print(f"样本内R²: {r2_in_ols_rent:.4f}")  # 打印R²
mae_in_ols_rent, rmse_in_ols_rent = evaluate_model_safe(y_train_rent_log, y_train_pred_log_ols_rent, "Rent OLS In-sample")

# --- 4. 样本外评估 ---
print("\n--- (B) 样本外 (Out-of-sample) 评估 (Rent OLS) ---")
y_val_pred_log_ols_rent = ols_model_rent.predict(X_val_scaled_df_rent)
# 新增：计算样本外R²
r2_out_ols_rent = r2_score(y_val_rent_log, y_val_pred_log_ols_rent)
print(f"样本外R²: {r2_out_ols_rent:.4f}")  # 打印R²
mae_out_ols_rent, rmse_out_ols_rent = evaluate_model_safe(y_val_rent_log, y_val_pred_log_ols_rent, "Rent OLS Out-of-sample")

# --- 5. 准备记录结果 ---
model_performance_rent = pd.DataFrame(index=[
    '样本内R²', '样本外R²',  # 新增R²指标
    'MAE_in', 'RMSE_in', 'MAE_out', 'RMSE_out'
])
model_performance_rent['OLS'] = [
    r2_in_ols_rent, r2_out_ols_rent,  # 存入R²
    mae_in_ols_rent, rmse_in_ols_rent, mae_out_ols_rent, rmse_out_ols_rent
]
print("\n--- Rent OLS 模型性能已记录 ---")
print(model_performance_rent)

正在训练 OLS 模型 (使用 139 个特征)...
OLS 模型训练完毕。

--- (A) 样本内 (In-sample) 评估 (Rent OLS) ---
样本内R²: 0.8681
--- Rent OLS In-sample Performance ---
MAE (原始租金): 121,895.67
RMSE (原始租金): 284,595.85
MAE (对数尺度): 0.2013
RMSE (对数尺度): 0.2759

--- (B) 样本外 (Out-of-sample) 评估 (Rent OLS) ---
样本外R²: 0.8570
--- Rent OLS Out-of-sample Performance ---
MAE (原始租金): 124,446.07
RMSE (原始租金): 277,021.04
MAE (对数尺度): 0.2091
RMSE (对数尺度): 0.2858

--- Rent OLS 模型性能已记录 ---
                    OLS
样本内R²          0.868093
样本外R²          0.856988
MAE_in    121895.667231
RMSE_in   284595.854061
MAE_out   124446.065403
RMSE_out  277021.044480


In [27]:
# --- 租金步骤 9: LassoCV 调优 ---
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # 新增r2_score
from sklearn.linear_model import LassoCV
# (确保 evaluate_model_safe 函数已定义)
import numpy as np
import pandas as pd 
import warnings 
import time 

print("\n--- 步骤 9 (租金版)：使用 LassoCV 进行超参数调优 ---")

# --- 1. 定义 alpha 候选范围 ---
alphas_lasso_rent = np.logspace(-5, -1, 20) # 搜索更广、更小的值
print(f"Lasso (Rent) 将尝试以下 alpha 值 (约): {alphas_lasso_rent}")

# --- 2. 初始化并运行 LassoCV ---
lasso_cv_model_rent = LassoCV(alphas=alphas_lasso_rent, 
                              cv=6, # 6 折
                              random_state=111, 
                              n_jobs=4, 
                              max_iter=5000, 
                              tol=1e-3, 
                              verbose=0) # 可以设为 1 看进度

print(f"正在运行 LassoCV (Rent)...")
start_time = time.time() 
# --- 使用最终的租金数据 ---
lasso_cv_model_rent.fit(X_train_scaled_df_rent, y_train_rent_log) 
end_time = time.time() 
elapsed_time = end_time - start_time
print(f"LassoCV (Rent) 完成。耗时: {elapsed_time:.2f} 秒")

# --- 3. 获取最佳 alpha ---
best_alpha_lasso_rent = lasso_cv_model_rent.alpha_
print(f"LassoCV (Rent) 找到的最佳 alpha 是: {best_alpha_lasso_rent:.6f}")

# --- 4. 使用最佳 alpha 模型评估 ---
best_lasso_model_rent = lasso_cv_model_rent 
coeffs_best_lasso_rent = best_lasso_model_rent.coef_
num_non_zero_coeffs_best_rent = np.sum(coeffs_best_lasso_rent != 0)
print(f"Best Lasso (Rent) 选择了 {num_non_zero_coeffs_best_rent} 个特征 / {X_train_scaled_df_rent.shape[1]}。")

print("\n--- (A) 样本内评估 (Best Rent Lasso) ---")
y_train_pred_log_best_lasso_rent = best_lasso_model_rent.predict(X_train_scaled_df_rent)
# 新增：样本内R²
r2_in_lasso_rent = r2_score(y_train_rent_log, y_train_pred_log_best_lasso_rent)
print(f"样本内R²: {r2_in_lasso_rent:.4f}")
mae_in_best_lasso_rent, rmse_in_best_lasso_rent = evaluate_model_safe(y_train_rent_log, y_train_pred_log_best_lasso_rent, "Best Rent Lasso In-sample")

print("\n--- (B) 样本外评估 (Best Rent Lasso) ---")
y_val_pred_log_best_lasso_rent = best_lasso_model_rent.predict(X_val_scaled_df_rent) 
# 新增：样本外R²
r2_out_lasso_rent = r2_score(y_val_rent_log, y_val_pred_log_best_lasso_rent)
print(f"样本外R²: {r2_out_lasso_rent:.4f}")
mae_out_best_lasso_rent, rmse_out_best_lasso_rent = evaluate_model_safe(y_val_rent_log, y_val_pred_log_best_lasso_rent, "Best Rent Lasso Out-of-sample")

# --- 5. 更新性能记录表 ---
try:
    model_performance_rent[f'Lasso (alpha={best_alpha_lasso_rent:.4f})'] = [
    r2_in_lasso_rent, r2_out_lasso_rent,  # 存入R²
    mae_in_best_lasso_rent, rmse_in_best_lasso_rent,
    mae_out_best_lasso_rent, rmse_out_best_lasso_rent
]
except NameError: pass 

print("\n--- Best Rent Lasso 模型性能已记录 ---")
print(model_performance_rent)


--- 步骤 9 (租金版)：使用 LassoCV 进行超参数调优 ---
Lasso (Rent) 将尝试以下 alpha 值 (约): [1.00000000e-05 1.62377674e-05 2.63665090e-05 4.28133240e-05
 6.95192796e-05 1.12883789e-04 1.83298071e-04 2.97635144e-04
 4.83293024e-04 7.84759970e-04 1.27427499e-03 2.06913808e-03
 3.35981829e-03 5.45559478e-03 8.85866790e-03 1.43844989e-02
 2.33572147e-02 3.79269019e-02 6.15848211e-02 1.00000000e-01]
正在运行 LassoCV (Rent)...


/opt/conda/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:633: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 408.71136437868563, tolerance: 38.046056199936565
  model = cd_fast.enet_coordinate_descent_gram(
/opt/conda/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:633: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 397.8804641001525, tolerance: 38.144379672053866
  model = cd_fast.enet_coordinate_descent_gram(
/opt/conda/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:633: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 401.9466238519949, tolerance: 38.03793299612174
  model = cd_fast.enet_coordinate_descent_gram(
/opt/conda/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:633: ConvergenceWarn

LassoCV (Rent) 完成。耗时: 10.45 秒
LassoCV (Rent) 找到的最佳 alpha 是: 0.000483
Best Lasso (Rent) 选择了 95 个特征 / 139。

--- (A) 样本内评估 (Best Rent Lasso) ---
样本内R²: 0.8679
--- Best Rent Lasso In-sample Performance ---
MAE (原始租金): 121,994.72
RMSE (原始租金): 285,374.98
MAE (对数尺度): 0.2014
RMSE (对数尺度): 0.2760

--- (B) 样本外评估 (Best Rent Lasso) ---
样本外R²: 0.8654
--- Best Rent Lasso Out-of-sample Performance ---
MAE (原始租金): 120,925.73
RMSE (原始租金): 278,066.51
MAE (对数尺度): 0.2021
RMSE (对数尺度): 0.2773

--- Best Rent Lasso 模型性能已记录 ---
                    OLS  Lasso (alpha=0.0005)
样本内R²          0.868093              0.867930
样本外R²          0.856988              0.865395
MAE_in    121895.667231         121994.716360
RMSE_in   284595.854061         285374.976403
MAE_out   124446.065403         120925.728478
RMSE_out  277021.044480         278066.505356


In [28]:
# --- 租金步骤 10: RidgeCV 调优 ---
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # 新增r2_score
from sklearn.linear_model import RidgeCV
# (确保 evaluate_model_safe 函数已定义)
import numpy as np
import pandas as pd 
import warnings 
import time 

print("\n--- 步骤 10 (租金版)：使用 RidgeCV 进行超参数调优 ---")

# --- 1. 定义 alpha 候选范围 ---
alphas_ridge_rent = np.logspace(-1, 4, 20) # 0.1 到 10000
print(f"Ridge (Rent) 将尝试以下 alpha 值 (约): {alphas_ridge_rent}")

# --- 2. 初始化并运行 RidgeCV ---
ridge_cv_model_rent = RidgeCV(alphas=alphas_ridge_rent, 
                              cv=6, 
                              scoring=None) # 使用默认 R^2

print(f"正在运行 RidgeCV (Rent)...")
start_time = time.time() 
ridge_cv_model_rent.fit(X_train_scaled_df_rent, y_train_rent_log) 
end_time = time.time() 
elapsed_time = end_time - start_time
print(f"RidgeCV (Rent) 完成。耗时: {elapsed_time:.2f} 秒")

# --- 3. 获取最佳 alpha ---
best_alpha_ridge_rent = ridge_cv_model_rent.alpha_
print(f"RidgeCV (Rent) 找到的最佳 alpha 是: {best_alpha_ridge_rent:.6f}")

# --- 4. 使用最佳 alpha 模型评估 ---
best_ridge_model_rent = ridge_cv_model_rent 

print("\n--- (A) 样本内评估 (Best Rent Ridge) ---")
y_train_pred_log_best_ridge_rent = best_ridge_model_rent.predict(X_train_scaled_df_rent)
r2_in_ridge_rent = r2_score(y_train_rent_log, y_train_pred_log_best_ridge_rent)
print(f"样本内R²: {r2_in_ridge_rent:.4f}")
mae_in_best_ridge_rent, rmse_in_best_ridge_rent = evaluate_model_safe(y_train_rent_log, y_train_pred_log_best_ridge_rent, "Best Rent Ridge In-sample")

print("\n--- (B) 样本外评估 (Best Rent Ridge) ---")
y_val_pred_log_best_ridge_rent = best_ridge_model_rent.predict(X_val_scaled_df_rent) 
r2_out_ridge_rent = r2_score(y_val_rent_log, y_val_pred_log_best_ridge_rent)
print(f"样本外R²: {r2_out_ridge_rent:.4f}")
mae_out_best_ridge_rent, rmse_out_best_ridge_rent = evaluate_model_safe(y_val_rent_log, y_val_pred_log_best_ridge_rent, "Best Rent Ridge Out-of-sample")

# --- 5. 更新性能记录表 ---
try:
    model_performance_rent[f'Ridge (alpha={best_alpha_ridge_rent:.2f})'] = [
    r2_in_ridge_rent, r2_out_ridge_rent,  # 存入R²
    mae_in_best_ridge_rent, rmse_in_best_ridge_rent,
    mae_out_best_ridge_rent, rmse_out_best_ridge_rent
]
except NameError: pass

print("\n--- Best Rent Ridge 模型性能已记录 ---")
print(model_performance_rent)


--- 步骤 10 (租金版)：使用 RidgeCV 进行超参数调优 ---
Ridge (Rent) 将尝试以下 alpha 值 (约): [1.00000000e-01 1.83298071e-01 3.35981829e-01 6.15848211e-01
 1.12883789e+00 2.06913808e+00 3.79269019e+00 6.95192796e+00
 1.27427499e+01 2.33572147e+01 4.28133240e+01 7.84759970e+01
 1.43844989e+02 2.63665090e+02 4.83293024e+02 8.85866790e+02
 1.62377674e+03 2.97635144e+03 5.45559478e+03 1.00000000e+04]
正在运行 RidgeCV (Rent)...
RidgeCV (Rent) 完成。耗时: 73.09 秒
RidgeCV (Rent) 找到的最佳 alpha 是: 42.813324

--- (A) 样本内评估 (Best Rent Ridge) ---
样本内R²: 0.8680
--- Best Rent Ridge In-sample Performance ---
MAE (原始租金): 121,881.22
RMSE (原始租金): 284,520.84
MAE (对数尺度): 0.2012
RMSE (对数尺度): 0.2759

--- (B) 样本外评估 (Best Rent Ridge) ---
样本外R²: 0.8657
--- Best Rent Ridge Out-of-sample Performance ---
MAE (原始租金): 120,798.83
RMSE (原始租金): 277,855.61
MAE (对数尺度): 0.2018
RMSE (对数尺度): 0.2769

--- Best Rent Ridge 模型性能已记录 ---
                    OLS  Lasso (alpha=0.0005)  Ridge (alpha=42.81)
样本内R²          0.868093              0.867930             0

In [29]:
# --- 租金步骤 11: ElasticNetCV 调优（修复 ConvergenceWarning 未定义错误） ---
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # 新增r2_score
from sklearn.linear_model import ElasticNetCV
from sklearn.exceptions import ConvergenceWarning  # 新增：导入收敛警告类
# (确保 evaluate_model_safe 函数已定义)
import numpy as np
import pandas as pd 
import warnings 
import time 

print("\n--- 步骤 11 (租金版)：使用 ElasticNetCV 进行超参数调优 ---")

# --- 1. 定义 alpha 和 l1_ratio 候选范围 ---
alphas_elastic_rent = np.logspace(-5, 2, 20)  # 涵盖更宽的正则化强度范围
l1_ratios_elastic_rent = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]  # 不同L1/L2比例
print(f"ElasticNet (Rent) 将尝试以下 alpha 值 (约): {alphas_elastic_rent}")
print(f"ElasticNet (Rent) 将尝试以下 l1_ratio 值: {l1_ratios_elastic_rent}")

# --- 2. 初始化并运行 ElasticNetCV（增加迭代次数减少收敛警告） ---
elastic_cv_model_rent = ElasticNetCV(alphas=alphas_elastic_rent,
                                     l1_ratio=l1_ratios_elastic_rent,
                                     cv=6,  # 保持6折交叉验证
                                     random_state=111,
                                     n_jobs=4,
                                     max_iter=10000,  # 提升迭代次数解决收敛问题
                                     tol=1e-3,
                                     verbose=0)  # 0为不输出进度信息

print(f"正在运行 ElasticNetCV (Rent)...")
start_time = time.time()
# 抑制收敛警告（现在可以正确识别 ConvergenceWarning 了）
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    elastic_cv_model_rent.fit(X_train_scaled_df_rent, y_train_rent_log)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"ElasticNetCV (Rent) 完成。耗时: {elapsed_time:.2f} 秒")

# --- 3. 获取最佳超参数 ---
best_alpha_elastic_rent = elastic_cv_model_rent.alpha_
best_l1_ratio_elastic_rent = elastic_cv_model_rent.l1_ratio_
print(f"ElasticNetCV (Rent) 找到的最佳 alpha 是: {best_alpha_elastic_rent:.6f}")
print(f"ElasticNetCV (Rent) 找到的最佳 l1_ratio 是: {best_l1_ratio_elastic_rent:.4f}")

# --- 4. 使用最佳模型评估 ---
best_elastic_model_rent = elastic_cv_model_rent
coeffs_best_elastic_rent = best_elastic_model_rent.coef_
num_non_zero_coeffs_elastic_rent = np.sum(coeffs_best_elastic_rent != 0)
print(f"Best ElasticNet (Rent) 选择了 {num_non_zero_coeffs_elastic_rent} 个特征 / {X_train_scaled_df_rent.shape[1]}。")

print("\n--- (A) 样本内评估 (Best Rent ElasticNet) ---")
y_train_pred_log_best_elastic_rent = best_elastic_model_rent.predict(X_train_scaled_df_rent)
r2_in_elastic_rent = r2_score(y_train_rent_log, y_train_pred_log_best_elastic_rent)
print(f"样本内R²: {r2_in_elastic_rent:.4f}")
mae_in_best_elastic_rent, rmse_in_best_elastic_rent = evaluate_model_safe(y_train_rent_log, y_train_pred_log_best_elastic_rent, "Best Rent ElasticNet In-sample")

print("\n--- (B) 样本外评估 (Best Rent ElasticNet) ---")
y_val_pred_log_best_elastic_rent = best_elastic_model_rent.predict(X_val_scaled_df_rent)
r2_out_elastic_rent = r2_score(y_val_rent_log, y_val_pred_log_best_elastic_rent)
print(f"样本外R²: {r2_out_elastic_rent:.4f}")
mae_out_best_elastic_rent, rmse_out_best_elastic_rent = evaluate_model_safe(y_val_rent_log, y_val_pred_log_best_elastic_rent, "Best Rent ElasticNet Out-of-sample")

# --- 5. 更新性能记录表（确保索引匹配） ---
try:
    # 统一索引为4个标准指标
    model_performance_rent = model_performance_rent.reindex(['样本内R²', '样本外R²', 'MAE_in', 'RMSE_in', 'MAE_out', 'RMSE_out'])
    # 插入ElasticNet结果
    model_performance_rent[f'ElasticNet (alpha={best_alpha_elastic_rent:.4f}, l1={best_l1_ratio_elastic_rent:.2f})'] = [
    r2_in_elastic_rent, r2_out_elastic_rent,  # 存入R²
    mae_in_best_elastic_rent, rmse_in_best_elastic_rent,
    mae_out_best_elastic_rent, rmse_out_best_elastic_rent
]
    print("\n--- Best Rent ElasticNet 模型性能已记录 ---")
    print(model_performance_rent)
except NameError:
    print("\n--- 警告：model_performance_rent 未定义，无法记录性能 ---")
except ValueError as e:
    print(f"\n--- 索引处理错误：{str(e)} ---")


--- 步骤 11 (租金版)：使用 ElasticNetCV 进行超参数调优 ---
ElasticNet (Rent) 将尝试以下 alpha 值 (约): [1.00000000e-05 2.33572147e-05 5.45559478e-05 1.27427499e-04
 2.97635144e-04 6.95192796e-04 1.62377674e-03 3.79269019e-03
 8.85866790e-03 2.06913808e-02 4.83293024e-02 1.12883789e-01
 2.63665090e-01 6.15848211e-01 1.43844989e+00 3.35981829e+00
 7.84759970e+00 1.83298071e+01 4.28133240e+01 1.00000000e+02]
ElasticNet (Rent) 将尝试以下 l1_ratio 值: [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]
正在运行 ElasticNetCV (Rent)...
ElasticNetCV (Rent) 完成。耗时: 39.57 秒
ElasticNetCV (Rent) 找到的最佳 alpha 是: 0.000695
ElasticNetCV (Rent) 找到的最佳 l1_ratio 是: 0.7000
Best ElasticNet (Rent) 选择了 95 个特征 / 139。

--- (A) 样本内评估 (Best Rent ElasticNet) ---
样本内R²: 0.8679
--- Best Rent ElasticNet In-sample Performance ---
MAE (原始租金): 121,989.92
RMSE (原始租金): 285,334.74
MAE (对数尺度): 0.2013
RMSE (对数尺度): 0.2760

--- (B) 样本外评估 (Best Rent ElasticNet) ---
样本外R²: 0.8654
--- Best Rent ElasticNet Out-of-sample Performance ---
MAE (原始租金): 120,925.57
RMSE (原始租金): 278,0

In [31]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, make_scorer, r2_score  # 新增r2_score
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet  # 新增ElasticNet
import numpy as np
import pandas as pd
import warnings
import os

print("\n--- 步骤 12 (租金版)：计算 6 折交叉验证 MAE 和 R² ---")  # 更新标题

# --- 1. 确保评估函数和评分器已定义 ---
def mae_original_scale_scorer(y_true_log, y_pred_log):
    try:
        y_true_orig = np.expm1(y_true_log)
        y_pred_orig = np.expm1(y_pred_log)
        if not np.all(np.isfinite(y_pred_orig)):
            return -np.inf 
        mae = mean_absolute_error(y_true_orig, y_pred_orig)
        return -mae 
    except (OverflowError, ValueError):
        return -np.inf

mae_scorer = make_scorer(mae_original_scale_scorer, greater_is_better=True)

# --- 2. 准备交叉验证 ---
kf = KFold(n_splits=6, shuffle=True, random_state=111) 
X_rent = X_train_scaled_df_rent  # 使用最终训练数据
y_rent = y_train_rent_log       # 使用最终训练目标

# --- 3. 计算 OLS 的 CV MAE ---
print("\n正在计算 Rent OLS 的 6 折 CV MAE...")
ols_model_for_cv_rent = LinearRegression()

# 设置临时解决方案：使用单线程避免编码问题
with warnings.catch_warnings(): 
    warnings.simplefilter("ignore")
    try:
        # 先尝试使用单线程
        ols_cv_scores_neg_mae_rent = cross_val_score(
            ols_model_for_cv_rent, X_rent, y_rent, cv=kf, 
            scoring=mae_scorer, n_jobs=1
        )
    except UnicodeEncodeError:
        # 如果还有编码问题，使用更保守的方法
        print("检测到编码问题，使用更安全的执行方式...")
        scores = []
        for train_idx, test_idx in kf.split(X_rent):
            X_train_fold, X_test_fold = X_rent.iloc[train_idx], X_rent.iloc[test_idx]
            y_train_fold, y_test_fold = y_rent.iloc[train_idx], y_rent.iloc[test_idx]
            
            model = LinearRegression()
            model.fit(X_train_fold, y_train_fold)
            y_pred_fold = model.predict(X_test_fold)
            
            score = mae_original_scale_scorer(y_test_fold, y_pred_fold)
            scores.append(score)
        
        ols_cv_scores_neg_mae_rent = np.array(scores)

ols_cv_scores_mae_rent = -ols_cv_scores_neg_mae_rent
ols_cv_mae_mean_rent = np.mean(ols_cv_scores_mae_rent[np.isfinite(ols_cv_scores_mae_rent)]) if np.any(np.isfinite(ols_cv_scores_mae_rent)) else np.nan
print(f"Rent OLS 6 折 CV MAE: {ols_cv_mae_mean_rent:,.2f}" if not np.isnan(ols_cv_mae_mean_rent) else "Rent OLS 6 折 CV MAE: N/A")

# --- 4. 计算 Best Lasso 的 CV MAE ---
print("\n正在计算 Best Rent Lasso (alpha={:.4f}) 的 6 折 CV MAE...".format(best_alpha_lasso_rent))
best_lasso_for_cv_rent = Lasso(alpha=best_alpha_lasso_rent, max_iter=5000, tol=1e-3, random_state=111) 

try:
    lasso_cv_scores_neg_mae_rent = cross_val_score(
        best_lasso_for_cv_rent, X_rent, y_rent, cv=kf, 
        scoring=mae_scorer, n_jobs=1
    )
except UnicodeEncodeError:
    print("检测到编码问题，使用更安全的执行方式...")
    scores = []
    for train_idx, test_idx in kf.split(X_rent):
        X_train_fold, X_test_fold = X_rent.iloc[train_idx], X_rent.iloc[test_idx]
        y_train_fold, y_test_fold = y_rent.iloc[train_idx], y_rent.iloc[test_idx]
        
        model = Lasso(alpha=best_alpha_lasso_rent, max_iter=5000, tol=1e-3, random_state=111)
        model.fit(X_train_fold, y_train_fold)
        y_pred_fold = model.predict(X_test_fold)
        
        score = mae_original_scale_scorer(y_test_fold, y_pred_fold)
        scores.append(score)
    
    lasso_cv_scores_neg_mae_rent = np.array(scores)

lasso_cv_mae_mean_rent = -np.mean(lasso_cv_scores_neg_mae_rent)
print(f"Best Rent Lasso 6 折 CV MAE: {lasso_cv_mae_mean_rent:,.2f}")

# --- 5. 计算 Best Ridge 的 CV MAE ---
print("\n正在计算 Best Rent Ridge (alpha={:.2f}) 的 6 折 CV MAE...".format(best_alpha_ridge_rent))
best_ridge_for_cv_rent = Ridge(alpha=best_alpha_ridge_rent, random_state=111)

try:
    ridge_cv_scores_neg_mae_rent = cross_val_score(
        best_ridge_for_cv_rent, X_rent, y_rent, cv=kf, 
        scoring=mae_scorer, n_jobs=1
    )
except UnicodeEncodeError:
    print("检测到编码问题，使用更安全的执行方式...")
    scores = []
    for train_idx, test_idx in kf.split(X_rent):
        X_train_fold, X_test_fold = X_rent.iloc[train_idx], X_rent.iloc[test_idx]
        y_train_fold, y_test_fold = y_rent.iloc[train_idx], y_rent.iloc[test_idx]
        
        model = Ridge(alpha=best_alpha_ridge_rent, random_state=111)
        model.fit(X_train_fold, y_train_fold)
        y_pred_fold = model.predict(X_test_fold)
        
        score = mae_original_scale_scorer(y_test_fold, y_pred_fold)
        scores.append(score)
    
    ridge_cv_scores_neg_mae_rent = np.array(scores)

ridge_cv_mae_mean_rent = -np.mean(ridge_cv_scores_neg_mae_rent)
print(f"Best Rent Ridge 6 折 CV MAE: {ridge_cv_mae_mean_rent:,.2f}")


# --- （新增开始）6. 计算 Best Rent ElasticNet 的 CV MAE ---
print("\n正在计算 Best Rent ElasticNet (alpha={:.6f}, l1={:.4f}) 的 6 折 CV MAE...".format(best_alpha_elastic_rent, best_l1_ratio_elastic_rent))
# 初始化ElasticNet模型（使用最佳超参数）
best_elastic_for_cv_mae_rent = ElasticNet(
    alpha=best_alpha_elastic_rent,  # 你的目标alpha=42.813324会自动从之前的ElasticNetCV结果继承
    l1_ratio=best_l1_ratio_elastic_rent,
    max_iter=10000,  # 避免收敛警告
    tol=1e-3,
    random_state=111
)

try:
    # 尝试用cross_val_score计算CV MAE（负分，需后续转换）
    elastic_cv_scores_neg_mae_rent = cross_val_score(
        best_elastic_for_cv_mae_rent, X_rent, y_rent, cv=kf, 
        scoring=mae_scorer, n_jobs=1
    )
except UnicodeEncodeError:
    # 编码错误时的备用方案：手动遍历每个折
    print("检测到编码问题，使用更安全的执行方式...")
    elastic_scores = []
    for train_idx, test_idx in kf.split(X_rent):
        X_train_fold, X_test_fold = X_rent.iloc[train_idx], X_rent.iloc[test_idx]
        y_train_fold, y_test_fold = y_rent.iloc[train_idx], y_rent.iloc[test_idx]
        
        model = ElasticNet(
            alpha=best_alpha_elastic_rent,
            l1_ratio=best_l1_ratio_elastic_rent,
            max_iter=10000,
            random_state=111
        )
        model.fit(X_train_fold, y_train_fold)
        y_pred_fold = model.predict(X_test_fold)
        
        score = mae_original_scale_scorer(y_test_fold, y_pred_fold)
        elastic_scores.append(score)
    
    elastic_cv_scores_neg_mae_rent = np.array(elastic_scores)

# 计算CV MAE均值（负分转正分）
elastic_cv_mae_mean_rent = -np.mean(elastic_cv_scores_neg_mae_rent)
print(f"Best Rent ElasticNet 6 折 CV MAE: {elastic_cv_mae_mean_rent:,.2f}")
# --- （新增结束）6. 计算 Best Rent ElasticNet 的 CV MAE ---


# --- （新增开始）计算交叉验证R² ---
# --- 新增：定义交叉验证R²的计算 ---
def cv_r2_score(model, X, y, cv):
    cv_r2_scores = cross_val_score(
        model, X, y, cv=cv, scoring='r2', n_jobs=1  # 使用r2评分器
    )
    return np.mean(cv_r2_scores)  # 返回R²均值

# --- 1. OLS的交叉验证R² ---
print("\n计算 Rent OLS 的 6 折 CV R²...")
ols_cv_r2_rent = cv_r2_score(ols_model_for_cv_rent, X_rent, y_rent, kf)
print(f"Rent OLS 交叉验证R²: {ols_cv_r2_rent:.4f}")

# --- 2. Lasso的交叉验证R² ---
print("\n计算 Best Rent Lasso 的 6 折 CV R²...")
lasso_cv_r2_rent = cv_r2_score(best_lasso_for_cv_rent, X_rent, y_rent, kf)
print(f"Best Rent Lasso 交叉验证R²: {lasso_cv_r2_rent:.4f}")

# --- 3. Ridge的交叉验证R² ---
print("\n计算 Best Rent Ridge 的 6 折 CV R²...")
ridge_cv_r2_rent = cv_r2_score(best_ridge_for_cv_rent, X_rent, y_rent, kf)
print(f"Best Rent Ridge 交叉验证R²: {ridge_cv_r2_rent:.4f}")

# --- 4. ElasticNet的交叉验证R² ---
print("\n计算 Best Rent ElasticNet 的 6 折 CV R²...")
best_elastic_for_cv_rent = ElasticNet(
    alpha=best_alpha_elastic_rent, 
    l1_ratio=best_l1_ratio_elastic_rent,
    max_iter=10000, 
    random_state=111
)
elastic_cv_r2_rent = cv_r2_score(best_elastic_for_cv_rent, X_rent, y_rent, kf)
print(f"Best Rent ElasticNet 交叉验证R²: {elastic_cv_r2_rent:.4f}")
# --- （新增结束）计算交叉验证R² ---


# --- 6. 存储 CV 结果 ---
try:
    if 'MAE_cv' in model_performance_rent.index:
        model_performance_rent = model_performance_rent.drop('MAE_cv')
    
    cv_results_rent = pd.Series({
        'OLS': ols_cv_mae_mean_rent,
        f'Lasso (alpha={best_alpha_lasso_rent:.4f})': lasso_cv_mae_mean_rent,
        f'Ridge (alpha={best_alpha_ridge_rent:.2f})': ridge_cv_mae_mean_rent,
        # 不再用np.nan，而是填入新增计算的ElasticNet CV MAE
        f'ElasticNet (alpha={best_alpha_elastic_rent:.4f}, l1={best_l1_ratio_elastic_rent:.2f})': elastic_cv_mae_mean_rent
    }, name='MAE_cv')
    
    model_performance_rent = pd.concat([model_performance_rent, cv_results_rent.to_frame().T])
    
    print("\n--- 租金交叉验证 MAE 已添加到性能表 ---")
    print(model_performance_rent.to_string(float_format='{:,.2f}'.format, na_rep='N/A'))
    
except NameError:
    print("\n错误：model_performance_rent 未定义。")
    # 创建新的性能表
    model_performance_rent = pd.DataFrame({
        'OLS': [ols_cv_mae_mean_rent],
        f'Lasso (alpha={best_alpha_lasso_rent:.4f})': [lasso_cv_mae_mean_rent],
        f'Ridge (alpha={best_alpha_ridge_rent:.2f})': [ridge_cv_mae_mean_rent],
        # 填入ElasticNet CV MAE
        f'ElasticNet (alpha={best_alpha_elastic_rent:.4f}, l1={best_l1_ratio_elastic_rent:.2f})': [elastic_cv_mae_mean_rent]
    }, index=['MAE_cv'])
    print("已创建新的性能表：")
    print(model_performance_rent.to_string(float_format='{:,.2f}'.format, na_rep='N/A'))


# --- （新增开始）汇总所有指标并输出结果 ---
# --- 更新性能表，加入交叉验证R² ---
model_performance_rent = model_performance_rent.reindex([
    '样本内R²', '样本外R²', '交叉验证R²',  # 新增交叉验证R²
    'MAE_in', 'RMSE_in', 'MAE_out', 'RMSE_out', 'MAE_cv'
])

# 填充交叉验证R²
model_performance_rent.loc['交叉验证R²', 'OLS'] = ols_cv_r2_rent
model_performance_rent.loc['交叉验证R²', f'Lasso (alpha={best_alpha_lasso_rent:.4f})'] = lasso_cv_r2_rent
model_performance_rent.loc['交叉验证R²', f'Ridge (alpha={best_alpha_ridge_rent:.2f})'] = ridge_cv_r2_rent
model_performance_rent.loc['交叉验证R²', f'ElasticNet (alpha={best_alpha_elastic_rent:.4f}, l1={best_l1_ratio_elastic_rent:.2f})'] = elastic_cv_r2_rent

# --- 打印最终性能对比表 ---
print("\n===== 租金模型综合性能比较 =====")
print(model_performance_rent.round(4))  # 保留4位小数
# --- （新增结束）汇总所有指标并输出结果 ---


# --- （新增开始）确定最佳模型 ---
# 提取关键指标（样本外R²最高、验证集RMSE最低优先）
candidates = model_performance_rent.columns
best_model = None
best_r2 = -np.inf
best_rmse = np.inf

for model in candidates:
    # 跳过全NaN的模型
    if model_performance_rent[model].isna().all():
        continue
    # 获取指标（处理可能的NaN）
    r2 = model_performance_rent.loc['样本外R²', model] if not pd.isna(model_performance_rent.loc['样本外R²', model]) else -np.inf
    rmse = model_performance_rent.loc['RMSE_out', model] if not pd.isna(model_performance_rent.loc['RMSE_out', model]) else np.inf
    
    # 优先样本外R²，其次RMSE
    if (r2 > best_r2) or (r2 == best_r2 and rmse < best_rmse):
        best_r2 = r2
        best_rmse = rmse
        best_model = model

print("\n===== 最佳模型 =====")
print(f"模型名称: {best_model}")
print(f"样本外R²: {best_r2:.4f}")
print(f"验证集RMSE: {best_rmse:.4f}")
print(f"验证集MAE: {model_performance_rent.loc['MAE_out', best_model]:.4f}")
print(f"交叉验证R²: {model_performance_rent.loc['交叉验证R²', best_model]:.4f}")
# --- （新增结束）确定最佳模型 ---


--- 步骤 12 (租金版)：计算 6 折交叉验证 MAE 和 R² ---

正在计算 Rent OLS 的 6 折 CV MAE...
Rent OLS 6 折 CV MAE: N/A

正在计算 Best Rent Lasso (alpha=0.0005) 的 6 折 CV MAE...
Best Rent Lasso 6 折 CV MAE: 122,245.26

正在计算 Best Rent Ridge (alpha=42.81) 的 6 折 CV MAE...
Best Rent Ridge 6 折 CV MAE: 122,245.97

正在计算 Best Rent ElasticNet (alpha=0.000695, l1=0.7000) 的 6 折 CV MAE...
Best Rent ElasticNet 6 折 CV MAE: 122,239.77

计算 Rent OLS 的 6 折 CV R²...
Rent OLS 交叉验证R²: -1830610061236888999559168.0000

计算 Best Rent Lasso 的 6 折 CV R²...
Best Rent Lasso 交叉验证R²: 0.8675

计算 Best Rent Ridge 的 6 折 CV R²...
Best Rent Ridge 交叉验证R²: 0.8674

计算 Best Rent ElasticNet 的 6 折 CV R²...
Best Rent ElasticNet 交叉验证R²: 0.8675

--- 租金交叉验证 MAE 已添加到性能表 ---
                                           OLS  Lasso (alpha=0.0005)  Ridge (alpha=42.81)  ElasticNet (alpha=0.0007, l1=0.70)
样本内R²                                     0.87                  0.87                 0.87                                0.87
样本外R²                                   

In [16]:
print("--- 步骤 13 (租金版)：生成 Kaggle 提交文件 (Rent Prediction) ---")

# --- 1. 确认测试集 ID 是否可用 ---
# (test_ids_rent 应该在之前的 "基础清洗与分割" 步骤中被存储了)
try:
    if 'test_ids_rent' not in locals() or test_ids_rent is None:
        # 如果 test_ids_rent 不存在，尝试重新加载原始测试文件获取
        print("警告：test_ids_rent 未找到，尝试重新加载...")
        original_test_filename = 'ruc_Class25Q2_test_rent.csv' # ！！确保文件名正确！！
        df_test_original = pd.read_csv(original_test_filename)
        if 'ID' in df_test_original.columns:
            test_ids_rent = df_test_original['ID']
            print(f"成功从 '{original_test_filename}' 加载了 {len(test_ids_rent)} 个测试 ID。")
            # 再次验证 ID 数量
            if len(test_ids_rent) != X_test_scaled_df_rent.shape[0]:
                 print(f"🚨 警告：ID 数量 ({len(test_ids_rent)}) 与测试集行数 ({X_test_scaled_df_rent.shape[0]}) 不匹配！")
        else:
            raise KeyError("原始测试集中未找到 'ID' 列")
            
    print(f"使用 {len(test_ids_rent)} 个测试 ID。")

except NameError:
    print(f"🚨 错误：变量 test_ids_rent 未定义。请确保步骤 2&3 (基础清洗与分割) 已成功运行。")
    # 停止
except FileNotFoundError:
    print(f"🚨 错误：无法在重新加载时找到原始 Kaggle 测试文件。请检查文件名。")
    # 停止
except KeyError as e:
     print(f"🚨 错误: {e}") 
     # 停止
except Exception as e:
     print(f"获取测试 ID 时出错: {e}")
     # 停止

# --- 2. 使用最佳模型 (Best Ridge) 进行预测 (在对数尺度上) ---
try:
    # 确保 best_ridge_model_rent 是我们之前训练好的 Ridge(alpha=42.81)
    if 'best_ridge_model_rent' not in locals(): raise NameError("best_ridge_model_rent 未定义")
    # 确保 X_test_scaled_df_rent 是最终的缩放后测试数据
    if 'X_test_scaled_df_rent' not in locals(): raise NameError("X_test_scaled_df_rent 未定义")
    
    print("正在使用 Best Ridge 模型对 Kaggle 测试集进行租金预测...")
    y_test_pred_log_rent = best_ridge_model_rent.predict(X_test_scaled_df_rent)
    print("预测完成 (对数尺度)。")

except NameError as e:
     print(f"🚨 错误：模型或测试数据变量未找到 ({e})。请确保之前的模型训练和数据准备步骤已成功运行。")
     # 停止
except Exception as e:
     print(f"预测过程中发生错误: {e}")
     # 停止

# --- 3. 转换回原始租金尺度 ---
print("正在将预测结果转换回原始租金尺度...")
try:
    # 使用 np.expm1()
    y_test_pred_orig_rent = np.expm1(y_test_pred_log_rent)
    print("转换完成。")
except NameError:
     print("🚨 错误：y_test_pred_log_rent 未定义。")
     # 停止
except Exception as e:
     print(f"转换回原始尺度时出错: {e}")
     # 停止
     
# --- 4. 创建提交 DataFrame ---
# (假设 Kaggle 仍然期望列名为 'Price', 即使是租金预测)
submission_col_name = 'Price' 
print(f"创建提交文件，预测列名为 '{submission_col_name}'...")

try:
    if len(test_ids_rent) == len(y_test_pred_orig_rent):
        submission_df_rent = pd.DataFrame({'ID': test_ids_rent, submission_col_name: y_test_pred_orig_rent})
        print("\n已创建租金提交 DataFrame:")
        print(submission_df_rent.head())

        # --- 5. 保存为 prediction_rent.csv 文件 ---
        submission_filename_rent = 'prediction_rent.csv'
        submission_df_rent.to_csv(submission_filename_rent, index=False)
        print(f"\n租金预测结果已成功保存到 '{submission_filename_rent}'！")
        print("你可以将这个文件提交到 Kaggle。")

    else:
        print(f"🚨 错误：ID 数量 ({len(test_ids_rent)}) 和预测数量 ({len(y_test_pred_orig_rent)}) 不匹配，无法创建提交文件。")

except NameError:
     print("🚨 错误：test_ids_rent 或 y_test_pred_orig_rent 未定义。")
except Exception as e:
     print(f"创建或保存 DataFrame 时出错: {e}")

--- 步骤 12 (租金版)：生成 Kaggle 提交文件 (Rent Prediction) ---
使用 9773 个测试 ID。
正在使用 Best Ridge 模型对 Kaggle 测试集进行租金预测...
预测完成 (对数尺度)。
正在将预测结果转换回原始租金尺度...
转换完成。
创建提交文件，预测列名为 'Price'...

已创建租金提交 DataFrame:
        ID         Price
0  2000000  1.666597e+05
1  2000001  3.552475e+05
2  2000002  4.093343e+05
3  2000003  1.609689e+06
4  2000004  1.173235e+06

租金预测结果已成功保存到 'prediction_rent.csv'！
你可以将这个文件提交到 Kaggle。


In [17]:
import pandas as pd

print("--- 开始合并预测文件 ---")

# --- 1. 定义文件名 ---
price_prediction_file = 'prediction.csv'
rent_prediction_file = 'prediction_rent.csv'
merged_output_file = 'merged_prediction.csv'

try:
    # --- 2. 加载两个 CSV 文件 ---
    print(f"正在加载房价预测文件: {price_prediction_file}...")
    df_price = pd.read_csv(price_prediction_file)
    print(f"  成功加载 {len(df_price)} 行。")

    print(f"正在加载租金预测文件: {rent_prediction_file}...")
    df_rent = pd.read_csv(rent_prediction_file)
    print(f"  成功加载 {len(df_rent)} 行。")

    # --- 3. 检查列名是否一致 (可选但推荐) ---
    if list(df_price.columns) != list(df_rent.columns):
        print("🚨 警告：两个文件的列名不完全一致！")
        print(f"  房价文件列名: {list(df_price.columns)}")
        print(f"  租金文件列名: {list(df_rent.columns)}")
        # 你可能需要在这里停止或手动统一列名

    # --- 4. 按顺序合并 DataFrame ---
    # pd.concat 会将 DataFrame 垂直堆叠
    # ignore_index=True 会创建一个新的连续索引 (0, 1, 2...)
    print("\n正在合并两个 DataFrame (房价在前，租金在后)...")
    merged_df = pd.concat([df_price, df_rent], ignore_index=True)
    print("合并完成。")

    # --- 5. 检查合并后的结果 (可选) ---
    print(f"\n合并后的 DataFrame 维度: {merged_df.shape}") # 行数应为两者之和
    print("合并后的 DataFrame (前 3 行):")
    print(merged_df.head(3))
    print("合并后的 DataFrame (后 3 行):")
    print(merged_df.tail(3))

    # --- 6. 保存合并后的 DataFrame 到新 CSV 文件 ---
    print(f"\n正在将合并结果保存到: {merged_output_file}...")
    # index=False 确保不将 DataFrame 的索引写入 CSV 文件
    merged_df.to_csv(merged_output_file, index=False)
    print(f"合并后的预测已成功保存到 '{merged_output_file}'！")

except FileNotFoundError:
    print(f"🚨 错误：找不到文件 '{price_prediction_file}' 或 '{rent_prediction_file}'。请确保文件在当前目录下且文件名正确。")
except Exception as e:
    print(f"合并或保存文件时发生错误: {e}")

--- 开始合并预测文件 ---
正在加载房价预测文件: prediction.csv...
  成功加载 34017 行。
正在加载租金预测文件: prediction_rent.csv...
  成功加载 9773 行。

正在合并两个 DataFrame (房价在前，租金在后)...
合并完成。

合并后的 DataFrame 维度: (43790, 2)
合并后的 DataFrame (前 3 行):
        ID        Price
0  1000000  19053712.30
1  1000001   2925502.43
2  1000002   3917970.38
合并后的 DataFrame (后 3 行):
            ID          Price
43787  2009770  233048.078099
43788  2009771  611703.327087
43789  2009772  901121.977648

正在将合并结果保存到: merged_prediction.csv...
合并后的预测已成功保存到 'merged_prediction.csv'！
